# 03 — Index A (Alpha-Tech)

**Objetivo.** Predecir Index_A con el mejor enfoque disponible, validado a 252 días de backtest autorregresivo.

**Pistas del enunciado.** Alpha-Tech — alta volatilidad, alto crecimiento. **Domina el RMSE junto a F** — aquí se gana o se pierde la competición.

**Nivel de esfuerzo.** ALTO. Explorar LSTM y CNN-LSTM. Ensemble de semillas sobre el ganador.

**Estrategia.** LSTM en log-ret mode → backtest → CNN-LSTM → backtest → ensemble 5 seeds del ganador → backtest. Quedarse con el que tenga menor RMSE en backtest 252d (nunca en loss de entrenamiento).

**Qué esperamos.** Alta volatilidad → error alto en cualquier caso. El ensemble de seeds reduce varianza. Comparar siempre con baseline drift (puede sorprender en series con tendencia fuerte).

**Qué NO hace.** No toca otros índices. No genera submission.

**Inputs.** `data/train_indices.csv`, `results/baselines.json`

**Output esperado.** `models/{OWNER}_Index_A.keras`, `results/index_A.json`

## 0. GPU workaround + imports + OWNER

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

OWNER = "oscar"   # <-- cada miembro pone su nombre aquí, UNA SOLA VEZ
IDX   = "Index_A"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    load_hackathon_data, make_window_dataset, make_temporal_split,
    build_model, train_model, train_ensemble,
    backtest_autoregressive, plot_history, plot_rollout,
    DATA_DIR, VAL_DAYS, V_IN_SHARED, RANDOM_SEED
)

os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)

MODEL_PATH = f'models/{OWNER}_{IDX}.keras'

## 1. Carga y diagnóstico de Index_A

In [ ]:
data = load_hackathon_data(DATA_DIR)
idx  = data['train_indices']
serie = idx[IDX].dropna().values

print(f'{IDX}: {len(serie)} días  |  min={serie.min():.0f}  max={serie.max():.0f}  último={serie[-1]:.0f}')

plt.figure(figsize=(13, 3))
plt.plot(serie, lw=0.8)
plt.title(f'{IDX} — precios brutos')
plt.tight_layout()
plt.show()

## 2. Referencia: RMSE de baselines para A

Este es el suelo a batir. Si ningún modelo lo supera en backtest, el baseline gana.

In [ ]:
with open('results/baselines.json') as f:
    baselines = json.load(f)

print(f'Baselines para {IDX}:')
for bl, rmse in baselines[IDX].items():
    print(f'  {bl:<15} RMSE = {rmse:,.0f}')

## 3. Preparación de datos — ventana deslizante en log-ret mode

`make_temporal_split` separa los últimos VAL_DAYS como validación. El dataset de ventanas se construye sobre la parte de entrenamiento. El split 80/20 interno es solo para que `train_model` tenga val_loss y active los callbacks.

In [ ]:
serie_train, _ = make_temporal_split(serie, val_days=VAL_DAYS, v_in=V_IN_SHARED)
X, y = make_window_dataset(serie_train, V_IN_SHARED, use_log_rets=True)

cut = int(len(X) * 0.8)
X_tr, y_tr = X[:cut], y[:cut]
X_v,  y_v  = X[cut:], y[cut:]

print(f'X_tr={X_tr.shape}  X_v={X_v.shape}  n_features={X.shape[2]}')

## 4. Candidato 1 — LSTM

In [ ]:
import tensorflow as tf
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model_lstm = build_model('lstm', V_IN_SHARED, n_features=X.shape[2], dropout=0.2)
hist_lstm  = train_model(model_lstm, X_tr, y_tr, X_v, y_v, epochs=300)
plot_history(hist_lstm, title=f'{IDX} — LSTM')

## 5. Backtest LSTM a 252 días

**Este número — no el loss de entrenamiento — es el que decide.**

In [ ]:
predict_fn_lstm = lambda x: model_lstm.predict(x, verbose=0).ravel()[0]
bt_lstm = backtest_autoregressive(predict_fn_lstm, serie, log_ret_mode=True)
print(f'LSTM  RMSE backtest = {bt_lstm["rmse"]:,.0f}  |  dir_acc = {bt_lstm["dir_accuracy"]:.2%}')
plot_rollout(serie, bt_lstm['preds'], index_name=f'{IDX} — LSTM', val_days=VAL_DAYS)

## 6. Candidato 2 — CNN-LSTM

In [ ]:
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model_cnn = build_model('cnn_lstm', V_IN_SHARED, n_features=X.shape[2])
hist_cnn  = train_model(model_cnn, X_tr, y_tr, X_v, y_v, epochs=300)
plot_history(hist_cnn, title=f'{IDX} — CNN-LSTM')

In [ ]:
predict_fn_cnn = lambda x: model_cnn.predict(x, verbose=0).ravel()[0]
bt_cnn = backtest_autoregressive(predict_fn_cnn, serie, log_ret_mode=True)
print(f'CNN-LSTM  RMSE backtest = {bt_cnn["rmse"]:,.0f}  |  dir_acc = {bt_cnn["dir_accuracy"]:.2%}')

## 7. Ensemble de semillas del ganador

`train_ensemble` entrena n_seeds modelos independientes y devuelve una `predict_fn` que promedia sus salidas.

In [ ]:
# Cambiar 'lstm' por 'cnn_lstm' si ese fue el ganador
best_tipo = 'lstm'

ens = train_ensemble(
    best_tipo, X_tr, y_tr, X_v, y_v,
    v_in=V_IN_SHARED, n_features=X.shape[2],
    n_seeds=5, epochs=300
)
print(f'Ensemble RMSE val (ventana, orientativo) = {ens["rmse_val"]:,.0f}')

In [ ]:
bt_ens = backtest_autoregressive(ens['predict_fn'], serie, log_ret_mode=True)
print(f'Ensemble  RMSE backtest = {bt_ens["rmse"]:,.0f}  |  dir_acc = {bt_ens["dir_accuracy"]:.2%}')
plot_rollout(serie, bt_ens['preds'], index_name=f'{IDX} — Ensemble {best_tipo}', val_days=VAL_DAYS)

## 8. Decisión final y guardado

Elegir el enfoque con menor RMSE de backtest 252d. Guardar modelo y `results/index_A.json`.

**Ancla de reconstrucción (CRÍTICO):** `precio_inicial` = `serie[-1]` = último precio real de train. Este valor se usa automáticamente dentro de `backtest_autoregressive` y `predict_autoregressive`. El notebook 09 también lo derivará de `train_indices[IDX].iloc[-1]`.

In [ ]:
# Comparativa final
resultados = {
    'lstm':     bt_lstm['rmse'],
    'cnn_lstm': bt_cnn['rmse'],
    'ensemble': bt_ens['rmse'],
}
for k, v in resultados.items():
    print(f'  {k:<12}  RMSE = {v:,.0f}')

ganador = min(resultados, key=resultados.get)
print(f'\nGanador: {ganador}')

In [ ]:
# Guardar el modelo ganador (si es ensemble, guardar el primero de los modelos)
# Si el ganador es el ensemble: guardar uno de los modelos seed como representante
# — en el rollout de producción (09) se recrea el ensemble con la misma lógica
if ganador in ('lstm', 'cnn_lstm'):
    best_model = model_lstm if ganador == 'lstm' else model_cnn
    best_model.save(MODEL_PATH)
    best_approach = 'nn'
    best_rmse     = resultados[ganador]
else:
    # Ensemble: guardar el primer modelo seed; indicar approach_type='nn_ensemble'
    ens['models'][0].save(MODEL_PATH)
    best_approach = 'nn_ensemble'
    best_rmse     = resultados['ensemble']

info = {
    'index':              IDX,
    'owner':              OWNER,
    'approach_type':      best_approach,
    'strategy':           ganador,
    'rmse_backtest_252d': best_rmse,
    'model_path':         MODEL_PATH,
    'log_ret_mode':       True,
    'v_in':               V_IN_SHARED,
    'ghost_source_index': None,
    'ghost_lag':          None,
    'notes':              f'{ganador}, dropout=0.2, 300 epochs'
}

with open(f'results/index_A.json', 'w') as f:
    json.dump(info, f, indent=2)

print('Guardado:', MODEL_PATH)
print('Guardado: results/index_A.json')
print(json.dumps(info, indent=2))